# 02 · Treino

Comparamos **3 modelos** com os mesmos dados e a mesma métrica, registramos tudo no
**MLflow** e só promovemos o vencedor se ele for **melhor que o campeão atual**.

| Modelo | Ideia |
|---|---|
| **Popularidade** (baseline) | as matérias mais lidas |
| **Item-CF** | "quem leu X também leu Y" (cosseno entre matérias) |
| **Híbrido** | item-CF + **conteúdo** (título e editoria) + recência → recomenda até matéria nova |

**Métricas (@10):** NDCG (acertar no topo vale mais) · HitRate (acertou pelo menos uma?) ·
acerto em **matérias frias** (sem nenhum clique no treino - o *cold start*).

In [ ]:
# Parâmetro do Job (rodando à mão, fica o valor padrão)
dbutils.widgets.text("schema", "news_recsys_dev")
SCHEMA = f"workspace.{dbutils.widgets.get('schema')}"
print(SCHEMA)

In [ ]:
import mlflow
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

MODELO = f"{SCHEMA}.news_recommender"
usuario = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{usuario}/{dbutils.widgets.get('schema')}")  # 1 por ambiente
mlflow.set_registry_uri("databricks-uc")

materias = spark.table(f"{SCHEMA}.bronze_articles").toPandas()
cliques = spark.table(f"{SCHEMA}.silver_clicks").toPandas()
print(len(materias), "matérias ·", len(cliques), "cliques")

## 1. Treino Vs teste (split temporal)

Para cada usuário, os **20% cliques mais recentes** viram teste. Nunca aleatório: em
produção o modelo sempre prevê o *futuro*.

In [ ]:
def split_temporal(cliques, frac_teste=0.2):
    cliques = cliques.sort_values(["user_id", "event_ts"])
    posicao = cliques.groupby("user_id").cumcount(ascending=False)  # 0 = mais recente
    total = cliques.groupby("user_id")["url"].transform("size")
    teste = posicao < (total * frac_teste).round().clip(lower=1)
    return cliques[~teste], cliques[teste]


def matriz(cliques, usuarios, urls):
    # matriz usuário Vs matéria: 1 = leu
    linhas = cliques["user_id"].map({u: i for i, u in enumerate(usuarios)})
    colunas = cliques["url"].map({u: i for i, u in enumerate(urls)})
    return sparse.csr_matrix((np.ones(len(cliques)), (linhas, colunas)),
                             shape=(len(usuarios), len(urls)))


treino, teste = split_temporal(cliques)
usuarios, urls = sorted(cliques["user_id"].unique()), materias["url"].tolist()
X = matriz(treino, usuarios, urls)
frias = set(urls) - set(treino["url"])
print(f"treino {len(treino)} · teste {len(teste)} · matérias sem clique no treino: {len(frias)}")

## 2. Os 3 modelos (cada um devolve um score para cada usuário Vs matéria)

In [ ]:
def score_popularidade(X, materias):
    return np.tile(np.asarray(X.sum(axis=0)).ravel(), (X.shape[0], 1))


def score_item_cf(X, materias):
    Xn = normalize(X, axis=0)  # cada matéria vira um vetor unitário
    similaridade = (Xn.T @ Xn).toarray()  # cosseno entre matérias
    np.fill_diagonal(similaridade, 0)
    return X @ similaridade


def score_hibrido(X, materias, alpha=0.4):
    texto = materias["title"] + " " + materias["section"]
    T = normalize(TfidfVectorizer().fit_transform(texto))
    publicada = pd.to_datetime(materias["published_at"], utc=True, format="mixed")
    idade_h = (publicada.max() - publicada).dt.total_seconds().to_numpy() / 3600
    recencia = 0.5 ** (idade_h / 24)  # notícia perde valor: meia-vida de 24 h
    conteudo = (normalize(X @ T) @ T.T).toarray() * recencia  # perfil Vs matéria
    cf = score_item_cf(X, materias)
    escala = lambda s: s / np.maximum(s.max(axis=1, keepdims=True), 1e-9)  # noqa: E731
    return alpha * escala(cf) + (1 - alpha) * escala(conteudo)


MODELOS = {"popularidade": score_popularidade, "item_cf": score_item_cf, "hibrido": score_hibrido}

In [ ]:
def top_k(scores, X, k=10):
    scores = scores.astype(float).copy()
    scores[X.toarray() > 0] = -np.inf  # não recomenda o que já leu
    return np.argsort(-scores, axis=1)[:, :k]


def avaliar(recs, teste, usuarios, urls, frias):
    lidas = teste.groupby("user_id")["url"].apply(set)
    ndcg, hit, hit_fria = [], [], []
    for i, u in enumerate(usuarios):
        if u not in lidas:
            continue
        rec = [urls[j] for j in recs[i]]
        ganhos = [1 / np.log2(r + 2) for r, url in enumerate(rec) if url in lidas[u]]
        ideal = sum(1 / np.log2(r + 2) for r in range(min(len(lidas[u]), 10)))
        ndcg.append(sum(ganhos) / ideal)
        hit.append(float(bool(ganhos)))
        if lidas[u] & frias:
            hit_fria.append(float(bool(set(rec) & lidas[u] & frias)))
    return {"ndcg_10": np.mean(ndcg), "hit_10": np.mean(hit), "hit_fria_10": np.mean(hit_fria)}

## 3. Treinar, medir e registrar no MLflow

Cada modelo vira uma **run** no MLflow (menu **Experiments**). Lá dá para comparar
as métricas lado a lado.

In [ ]:
resultados = {}
for nome, score in MODELOS.items():
    with mlflow.start_run(run_name=nome) as run:
        metricas = avaliar(top_k(score(X, materias), X), teste, usuarios, urls, frias)
        mlflow.log_param("modelo", nome)
        mlflow.log_metrics(metricas)
        resultados[nome] = {**metricas, "run_id": run.info.run_id}

tabela = pd.DataFrame(resultados).T.drop(columns="run_id").astype(float).round(3)
vencedor = tabela["ndcg_10"].idxmax()
print("vencedor:", vencedor)
tabela.sort_values("ndcg_10", ascending=False)

## 4. Empacotar o vencedor como modelo MLflow e registrar no Unity Catalog

O vencedor é re-treinado com **todos** os cliques (o teste era só para medir) e vira
um modelo `pyfunc`: entra um `user_id`, sai o top-10.

In [ ]:
class NewsRecommender(mlflow.pyfunc.PythonModel):
    def __init__(self, nome, X, usuarios, materias):
        self.nome, self.X, self.usuarios, self.materias = nome, X, usuarios, materias

    def predict(self, context, model_input):
        scores = MODELOS[self.nome](self.X, self.materias)
        recs = top_k(scores, self.X)
        indice = {u: i for i, u in enumerate(self.usuarios)}
        urls = self.materias["url"].tolist()
        linhas = [
            {"user_id": u, "rank": r + 1, "url": urls[j]}
            for u in model_input["user_id"] if u in indice
            for r, j in enumerate(recs[indice[u]])
        ]
        return pd.DataFrame(linhas)


todos = sorted(cliques["user_id"].unique())
modelo = NewsRecommender(vencedor, matriz(cliques, todos, urls), todos, materias)

with mlflow.start_run(run_id=resultados[vencedor]["run_id"]):
    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=modelo,
        input_example=pd.DataFrame({"user_id": todos[:2]}),
        registered_model_name=MODELO,
    )
versao = info.registered_model_version
print(f"{MODELO} versão {versao} registrada")

## 5. Quality gate: só vira `@champion` se for melhor

O pipeline re-treina todo dia sozinho. Se um dia o modelo novo for pior (dados
estranhos, bug), ele **não** substitui o campeão. O app e a inferência sempre usam o
alias `@champion` → rollback = mover o alias.

In [ ]:
client = MlflowClient()
ndcg_novo = resultados[vencedor]["ndcg_10"]
try:
    campeao = client.get_model_version_by_alias(MODELO, "champion")
    ndcg_campeao = client.get_run(campeao.run_id).data.metrics["ndcg_10"]
except Exception:  # ainda não existe campeão
    ndcg_campeao = None

promovido = ndcg_campeao is None or ndcg_novo > ndcg_campeao
if promovido:
    client.set_registered_model_alias(MODELO, "champion", versao)
texto_campeao = "nenhum" if ndcg_campeao is None else f"{ndcg_campeao:.3f}"
print(f"NDCG novo {ndcg_novo:.3f} Vs campeão {texto_campeao} → "
      + ("PROMOVIDO a @champion" if promovido else "mantém o campeão"))

historico = tabela.reset_index(names="modelo").assign(
    versao=str(versao), promovido=lambda d: promovido & (d["modelo"] == vencedor),
    avaliado_em=pd.Timestamp.now(),
)
spark.createDataFrame(historico).write.mode("append").saveAsTable(f"{SCHEMA}.model_metrics")

### O que percebemos?

- A popularidade perde feio: personalizar **vale a pena**.
- Item-CF não recomenda matéria nova (acerto em frias = 0); o híbrido sim.
- O MLflow guardou parâmetros, métricas e o modelo; o Unity Catalog guarda as versões.
- Rode o Job de novo com os mesmos dados: o NDCG empata e o modelo **não** é promovido.

**Próximo:** `03_recomendacoes`.